# COINS 01 - Full Dataset EDA

This notebook now lives inside `data/` because it documents the cached parquet files directly. It is only EDA: no noise injection, OOF scoring, repair, synthesis, or model training.


In [1]:
from __future__ import annotations

import os
import time
import warnings
from pathlib import Path
from typing import Any

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

import numpy as np
import pandas as pd
from scipy.stats import norm, wilcoxon
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

pd.set_option("display.max_columns", 160)
pd.set_option("display.max_rows", 260)
pd.options.mode.chained_assignment = None

ROOT = Path.cwd()
if not (ROOT / "data").exists() and ROOT.name == "data":
    ROOT = ROOT.parent
DATA_DIR = ROOT / "data"
RAW_RESULTS = ROOT / "docs" / "experiments" / "raw" / "full-benchmark-solution-v2.csv"
SMOKE = os.environ.get("COINS_SMOKE", "0") == "1"

DATA_FILES = {
    "pima": "pima.parquet", "credit-g": "credit-g.parquet", "yeast": "yeast.parquet",
    "ecoli": "ecoli.parquet", "phoneme": "phoneme.parquet", "breast_cancer": "breast_cancer.parquet",
    "ilpd": "ilpd.parquet", "blood": "blood.parquet", "haberman": "haberman.parquet",
    "ionosphere": "ionosphere.parquet", "vehicle_bus": "vehicle_bus.parquet", "glass_float": "glass_float.parquet",
    "abalone": "abalone.parquet", "spambase": "spambase.parquet", "kc1": "kc1.parquet",
}
MINORITY_TEXT = {
    "pima": "tested_positive", "credit-g": "bad", "yeast": "MIT", "ecoli": "im", "phoneme": "nasal",
    "breast_cancer": "malignant", "ilpd": "no_disease", "blood": "donated", "haberman": "died",
    "ionosphere": "bad", "vehicle_bus": "bus", "glass_float": "window_float", "abalone": "rings_gt_10",
    "spambase": "spam", "kc1": "defective",
}
SEEDS_FULL = [13, 17, 23, 29, 31, 37, 41, 43, 47, 53]
PROTOCOLS_FULL = {
    "hidden_minority_low": (0.10, 0.05),
    "hidden_minority_medium": (0.30, 0.10),
    "hidden_minority_high": (0.40, 0.20),
}
GOAL_MINORITY_RATIO = 0.15
BUDGET_RATIO = 0.10
DATA_NAMES = ["pima"] if SMOKE else list(DATA_FILES)
SEEDS = [13] if SMOKE else SEEDS_FULL
PROTOCOLS = {"hidden_minority_medium": PROTOCOLS_FULL["hidden_minority_medium"]} if SMOKE else PROTOCOLS_FULL
print("root", ROOT)
print("data_dir", DATA_DIR)
print("smoke", SMOKE)


root /home/than-minh/project/DSP
data_dir /home/than-minh/project/DSP/data
smoke False


In [2]:
def read_data(name: str) -> tuple[pd.DataFrame, np.ndarray, str]:
    df = pd.read_parquet(DATA_DIR / DATA_FILES[name])
    label_col = df.columns[-1]
    y = (df[label_col] == MINORITY_TEXT[name]).astype(int).to_numpy()
    X = df.drop(columns=[label_col])
    return X, y, label_col


## Cached Dataset Inventory

This table describes the raw parquet files before benchmark subsampling. `minority_text` is the original target value mapped to class `1`.


In [3]:
summary_rows = []
preview_tables = []
for name in DATA_NAMES:
    X, y, label_col = read_data(name)
    summary_rows.append({
        "dataset": name,
        "parquet_file": str(DATA_DIR / DATA_FILES[name]),
        "raw_rows": len(y),
        "raw_features": X.shape[1],
        "label_column": label_col,
        "minority_text": MINORITY_TEXT[name],
        "minority_rows": int((y == 1).sum()),
        "majority_rows": int((y == 0).sum()),
        "minority_ratio": float((y == 1).mean()),
        "numeric_features": int(len(X.select_dtypes(exclude=["object", "category", "bool"]).columns)),
        "categorical_features": int(len(X.select_dtypes(include=["object", "category", "bool"]).columns)),
    })
    preview = X.head(5).copy()
    preview.insert(0, "dataset", name)
    preview.insert(1, "binary_label", y[:5])
    preview_tables.append(preview)

dataset_eda = pd.DataFrame(summary_rows).sort_values("dataset").reset_index(drop=True)
display(dataset_eda)


,dataset,parquet_file,raw_rows,raw_features,label_column,minority_text,minority_rows,majority_rows,minority_ratio,numeric_features,categorical_features
0,abalone,/home/than-minh/project/DSP/data/abalone.parquet,2000,8,target,rings_gt_10,693,1307,0.346500,7,1
1,blood,/home/than-minh/project/DSP/data/blood.parquet,748,4,target,donated,178,570,0.237968,4,0
2,breast_cancer,/home/than-minh/project/DSP/data/breast_cancer...,569,30,target,malignant,212,357,0.372583,30,0
3,credit-g,/home/than-minh/project/DSP/data/credit-g.parquet,1000,20,target,bad,300,700,0.300000,7,13
4,ecoli,/home/than-minh/project/DSP/data/ecoli.parquet,336,7,target,im,77,259,0.229167,7,0
5,glass_float,/home/than-minh/project/DSP/data/glass_float.p...,214,9,target,window_float,70,144,0.327103,9,0
6,haberman,/home/than-minh/project/DSP/data/haberman.parquet,306,3,target,died,81,225,0.264706,2,1
7,ilpd,/home/than-minh/project/DSP/data/ilpd.parquet,583,10,target,no_disease,167,416,0.286449,9,1
8,ionosphere,/home/than-minh/project/DSP/data/ionosphere.pa...,351,34,target,bad,126,225,0.358974,34,0
9,kc1,/home/than-minh/project/DSP/data/kc1.parquet,2109,21,target,defective,326,1783,0.154576,21,0


## Raw `head(5)` Preview Per Dataset

Small previews prove every cached dataset is readable without filling the notebook with full raw tables.


In [4]:
for preview in preview_tables:
    print("DATASET", preview["dataset"].iloc[0])
    display(preview.head(5))


DATASET pima


,dataset,binary_label,preg,plas,pres,skin,insu,mass,pedi,age
0,pima,1,6.0,148.0,72.0,35.0,0.0,33.6,0.627,50.0
1,pima,0,1.0,85.0,66.0,29.0,0.0,26.6,0.351,31.0
2,pima,1,8.0,183.0,64.0,0.0,0.0,23.3,0.672,32.0
3,pima,0,1.0,89.0,66.0,23.0,94.0,28.1,0.167,21.0
4,pima,1,0.0,137.0,40.0,35.0,168.0,43.1,2.288,33.0


DATASET credit-g


,dataset,binary_label,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker
0,credit-g,0,<0,6,critical/other existing credit,radio/tv,1169.0,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes
1,credit-g,1,0<=X<200,48,existing paid,radio/tv,5951.0,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes
2,credit-g,0,no checking,12,critical/other existing credit,education,2096.0,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes
3,credit-g,0,<0,42,existing paid,furniture/equipment,7882.0,<100,4<=X<7,2,male single,guarantor,4,life insurance,45,none,for free,1,skilled,2,none,yes
4,credit-g,1,<0,24,delayed previously,new car,4870.0,<100,1<=X<4,3,male single,none,4,no known property,53,none,for free,2,skilled,2,none,yes


DATASET yeast


...

,dataset,binary_label,Sex,Length,Diameter,Height,Whole_weight,Shucked_weight,Viscera_weight,Shell_weight
0,abalone,0,I,0.315,0.230,0.080,0.1375,0.0545,0.0310,0.0445
1,abalone,0,I,0.455,0.355,0.105,0.3720,0.1380,0.0765,0.1350
2,abalone,0,I,0.450,0.330,0.105,0.4480,0.2080,0.0890,0.1200
3,abalone,0,F,0.570,0.465,0.160,0.8935,0.3145,0.2575,0.2630
4,abalone,0,I,0.530,0.430,0.160,0.7245,0.3210,0.1275,0.2400


DATASET spambase


,dataset,binary_label,word_freq_make,word_freq_address,word_freq_all,word_freq_3d,word_freq_our,word_freq_over,word_freq_remove,word_freq_internet,word_freq_order,word_freq_mail,word_freq_receive,word_freq_will,word_freq_people,word_freq_report,word_freq_addresses,word_freq_free,word_freq_business,word_freq_email,word_freq_you,word_freq_credit,word_freq_your,word_freq_font,word_freq_000,word_freq_money,word_freq_hp,word_freq_hpl,word_freq_george,word_freq_650,word_freq_lab,word_freq_labs,word_freq_telnet,word_freq_857,word_freq_data,word_freq_415,word_freq_85,word_freq_technology,word_freq_1999,word_freq_parts,word_freq_pm,word_freq_direct,word_freq_cs,word_freq_meeting,word_freq_original,word_freq_project,word_freq_re,word_freq_edu,word_freq_table,word_freq_conference,char_freq_%3B,char_freq_%28,char_freq_%5B,char_freq_%21,char_freq_%24,char_freq_%23,capital_run_length_average,capital_run_length_longest,capital_run_length_total
0,spambase,1,0.00,0.64,0.64,0.0,0.32,0.00,0.00,0.00,0.00,0.00,0.00,0.64,0.00,0.00,0.00,0.32,0.00,1.29,1.93,0.00,0.96,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.000,0.0,0.778,0.000,0.000,3.756,61,278
1,spambase,1,0.21,0.28,0.50,0.0,0.14,0.28,0.21,0.07,0.00,0.94,0.21,0.79,0.65,0.21,0.14,0.14,0.07,0.28,3.47,0.00,1.59,0.0,0.43,0.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.07,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.132,0.0,0.372,0.180,0.048,5.114,101,1028
2,spambase,1,0.06,0.00,0.71,0.0,1.23,0.19,0.19,0.12,0.64,0.25,0.38,0.45,0.12,0.00,1.75,0.06,0.06,1.03,1.36,0.32,0.51,0.0,1.16,0.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.06,0.0,0.0,0.12,0.0,0.06,0.06,0.0,0.0,0.01,0.143,0.0,0.276,0.184,0.010,9.821,485,2259
3,spambase,1,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,0.31,0.31,0.31,0.00,0.00,0.31,0.00,0.00,3.18,0.00,0.31,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.137,0.0,0.137,0.000,0.000,3.537,40,191
4,spambase,1,0.00,0.00,0.00,0.0,0.63,0.00,0.31,0.63,0.31,0.63,0.31,0.31,0.31,0.00,0.00,0.31,0.00,0.00,3.18,0.00,0.31,0.0,0.00,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.00,0.00,0.0,0.0,0.00,0.135,0.0,0.135,0.000,0.000,3.537,40,191


DATASET kc1


,dataset,binary_label,loc,v(g),ev(g),iv(g),n,v,l,d,i,e,b,t,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount
0,kc1,0,1.1,1.4,1.4,1.4,1.3,1.30,1.30,1.30,1.30,1.30,1.30,1.30,2.0,2,2,2,1.2,1.2,1.2,1.2,1.4
1,kc1,1,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.00,1.00,1.00,1.00,1.0,1,1,1,1.0,1.0,1.0,1.0,1.0
2,kc1,1,83.0,11.0,1.0,11.0,171.0,927.89,0.04,23.04,40.27,21378.61,0.31,1187.70,65.0,10,6,0,18.0,25.0,107.0,64.0,21.0
3,kc1,1,46.0,8.0,6.0,8.0,141.0,769.78,0.07,14.86,51.81,11436.73,0.26,635.37,37.0,2,5,0,16.0,28.0,89.0,52.0,15.0
4,kc1,1,25.0,3.0,1.0,3.0,58.0,254.75,0.11,9.35,27.25,2381.95,0.08,132.33,21.0,0,2,0,11.0,10.0,41.0,17.0,5.0
